# Guide-3_缓存&加载


In [ ]:
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "data" / "loader_roots").exists():
            return path
    raise FileNotFoundError("Could not find data/loader_roots from the current working directory.")


PROJECT_ROOT = find_project_root()
XJTU_ROOT = str(PROJECT_ROOT / "data" / "loader_roots" / "xjtu")
PHM2012_ROOT = str(PROJECT_ROOT / "data" / "loader_roots" / "phm2012")

from phm.data.labeler.BearingRulLabeler import BearingRulLabeler
from phm.data.loader.XJTULoader import XJTULoader
from phm.data.process.EntityPipeline import EntityPipeline
from phm.data.process.array.RMSProcessor import RMSProcessor
from phm.data.process.entity.ThreeSigmaFPTCalculator import ThreeSigmaFPTCalculator
from phm.engine.trainer.BaseTrainer import BaseTrainer
from phm.model.basic.MLP import MLP
from phm.util.Cache import Cache

# 读取原始数据
data_loader = XJTULoader(XJTU_ROOT)
bearing = data_loader("Bearing1_3", 'Horizontal Vibration')

# 提取特征
pipeline = EntityPipeline()
pipeline.step(
    entity=bearing,
    processor=RMSProcessor(data_loader['continuum']),
    input_key='Horizontal Vibration',
    output_key='H_RMS'
)
pipeline.step(
    entity=bearing,
    processor=ThreeSigmaFPTCalculator(),
    input_key='H_RMS',
)

# 构造数据集
labeler = BearingRulLabeler(2048, is_rectified=True, is_squeeze=True)
dataset = labeler(bearing, 'Horizontal Vibration')

# 缓存数据集
Cache.save(dataset, 'dataset')

# 加载数据集
dataset = Cache.load('dataset')

# 训练模型
trainer = BaseTrainer()
model = MLP(2048, 16, 1)
trainer.train(model, dataset)

# 缓存模型
Cache.save(model, 'model')

# 读取模型
model = Cache.load('model')
